In [1]:
from pathlib import Path

from astropy import table
from astropy.coordinates import SkyCoord

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import GaiaQuality, Local, LindegrenC1, LindegrenC2
from ugdatalab.methods.bayesian.likelihoods import LinearGaussianLikelihood
from ugdatalab.methods.bayesian.mixture import mixture_contamination

from plotters import (
    plot_mollweide_diff,
    plot_period_abs_mag,
    plot_period_abs_mag_c12_comparison,
    plot_inlier_prob_period_luminosity_comparison,
    plot_inlier_prob_map,
)


def pl_scatter_metrics(data, label):
    """Quick unweighted P-L scatter diagnostics."""
    periods = data["rrlyrae_representative_period"]
    m_g = data["M_G"]
    finite = np.isfinite(periods) & np.isfinite(m_g)
    x = np.log10(periods[finite])
    y = m_g[finite]
    A = np.column_stack([np.ones_like(x), x])
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    resid = y - A @ beta
    mad = np.median(np.abs(resid - np.median(resid))) * 1.4826
    return {
        "sample": label,
        "N": len(y),
        "sigma_MAD [mag]": round(mad, 3),
        "RMS [mag]": round(float(np.sqrt(np.mean(resid**2))), 3),
        "N(|ΔM| > 1 mag)": int(np.count_nonzero(np.abs(resid) > 1)),
        "N(|ΔM| > 2 mag)": int(np.count_nonzero(np.abs(resid) > 2)),
    }


def save_table_npz(path, data):
    """Save an astropy Table as a compressed .npz file."""
    path = Path(path)
    arrays = {col: np.array(data[col]) for col in data.colnames}
    np.savez_compressed(path, **arrays)
    return path

The archive is unstable and may perform below expectations. If launching multiple, consecutive, heavy queries through Python, please space them out (e.g., using sleep(1)) to avoid overloading the system. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the December 2025 infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


/Users/junruiting/GitHub/ay-128/.venv/lib/python3.14/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
"""


In [ ]:
rrlyrae_quality = GaiaQuality(query)


In [ ]:
rrlyrae_low_dust = Local(rrlyrae_quality)

In [ ]:
assignment_style_query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
WHERE gs.parallax_over_error > 5
  AND ABS(gs.b) > 30
  AND gs.parallax > 0.25
"""

assignment_style_query

In [ ]:
len(rrlyrae_low_dust.data)

In [ ]:
abs_b = np.abs(rrlyrae_low_dust.data["b"])

float(np.min(abs_b)), int(np.count_nonzero(abs_b < 30.0))

In [ ]:
c1 = LindegrenC1(rrlyrae_low_dust)
c12 = LindegrenC2(c1)
rrlyrae_calibration_data = c12.data

table.Table(
    rows=[
        {"sample": "Before C1/C2", "N": len(rrlyrae_low_dust.data)},
        {"sample": "Pass C1 and C2", "N": len(rrlyrae_calibration_data)},
    ]
)

In [ ]:
ax = plot_mollweide_diff(rrlyrae_low_dust.data, rrlyrae_calibration_data)
plt.show()

In [ ]:
ax = plot_period_abs_mag_c12_comparison(rrlyrae_low_dust.data, rrlyrae_calibration_data)
plt.show()

In [ ]:
scatter_metrics = table.Table(
    rows=[
        pl_scatter_metrics(rrlyrae_low_dust.data, "Before C1/C2"),
        pl_scatter_metrics(rrlyrae_calibration_data, "After C1/C2"),
    ]
)

scatter_metrics

In [ ]:
distance_kpc = 1.0 / rrlyrae_calibration_data["parallax"]

summary = table.Table(
    {
        "source_id": rrlyrae_calibration_data["source_id"],
        "best_classification": rrlyrae_calibration_data["best_classification"],
        "int_average_g": rrlyrae_calibration_data["int_average_g"],
        "b": rrlyrae_calibration_data["b"],
        "parallax": rrlyrae_calibration_data["parallax"],
        "parallax_error": rrlyrae_calibration_data["parallax_error"],
        "distance_kpc": distance_kpc,
    }
)

len(summary), summary[:10]

In [ ]:
prob_threshold = 0.95

# Fit mixture contamination per subclass
all_inlier_probs = np.full(len(rrlyrae_calibration_data), np.nan)
for rr_class in ["RRab", "RRc"]:
    mask = rrlyrae_calibration_data["best_classification"] == rr_class
    subset = rrlyrae_calibration_data[mask]
    likelihood = LinearGaussianLikelihood(
        x=np.log10(subset["rrlyrae_representative_period"]),
        y=subset["M_G"],
        y_err=subset["sigma_M"],
    )
    result = mixture_contamination(likelihood)
    all_inlier_probs[mask] = result.inlier_prob

# RRd: no mixture model, set to 1.0 (pass through)
rrd_mask = rrlyrae_calibration_data["best_classification"] == "RRd"
all_inlier_probs[rrd_mask] = 1.0

kept_mask = all_inlier_probs >= prob_threshold
rrlyrae_clean_data = rrlyrae_calibration_data[kept_mask]

table.Table(
    rows=[
        {"sample": "After C1/C2", "N": len(rrlyrae_calibration_data)},
        {"sample": rf"Mixture model ($p_{{\rm in}} \ge {prob_threshold:.2f}$)", "N": len(rrlyrae_clean_data)},
    ]
)

In [ ]:
cleaning_metrics = table.Table(
    rows=[
        pl_scatter_metrics(rrlyrae_calibration_data, "After C1/C2"),
        pl_scatter_metrics(rrlyrae_clean_data, "Mixture model"),
    ]
)

cleaning_metrics


In [ ]:
ax = plot_inlier_prob_period_luminosity_comparison(
    rrlyrae_calibration_data, all_inlier_probs, prob_threshold,
)
plt.show()

In [ ]:
ax = plot_inlier_prob_map(rrlyrae_calibration_data, all_inlier_probs)
ax.set_title(r"Calibration sample: inlier probability in the $P$--$M_G$ plane")
plt.show()

In [ ]:
distance_kpc = 1.0 / rrlyrae_clean_data["parallax"]

summary = table.Table(
    {
        "source_id": rrlyrae_clean_data["source_id"],
        "best_classification": rrlyrae_clean_data["best_classification"],
        "int_average_g": rrlyrae_clean_data["int_average_g"],
        "b": rrlyrae_clean_data["b"],
        "parallax": rrlyrae_clean_data["parallax"],
        "parallax_error": rrlyrae_clean_data["parallax_error"],
        "distance_kpc": distance_kpc,
    }
)

len(summary), summary[:10]


In [ ]:
ax = plot_period_abs_mag(rrlyrae_clean_data)
ax.set_title(rf"Final cleaned period-luminosity sample ($N={len(rrlyrae_clean_data)}$)")
plt.show()

In [ ]:
output_path = Path("rrlyrae_calibration_sample.npz")
save_table_npz(output_path, rrlyrae_clean_data)

{
    "path": str(output_path.resolve()),
    "N": len(rrlyrae_clean_data),
    "columns": len(rrlyrae_clean_data.colnames),
}